# Tutorial 4: Neural Networks in Practice (Fixed Split Evaluation)

This notebook is the hands-on companion to **Neural Networks and Deep Learning**.

## Learning goals
- Build an MLP from scratch in PyTorch
- Implement forward pass, loss, backpropagation, and optimizer updates
- Compare learning curves across hyperparameters
- Use a **fixed train/val/test split** for fair comparisons
- Export a standardized payload for class Pareto-front analysis

## Data contract

This notebook expects one data file and three split files:
- `dataset.csv` with columns:
  - `sample_id`
  - target column (default: `target`)
  - feature columns (recommended prefix: `fp_`)
- `train.csv`, `val.csv`, `test.csv` each containing a single `sample_id` column

Important: do not re-split data in this notebook.

In [ ]:
# If needed, install common dependencies.
# In most Colab runtimes these are already available.
# %pip install -q pandas numpy matplotlib scikit-learn torch

In [ ]:
import json
import time
import random
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Robust workspace setup for both Colab and local runs.
import os
import sys
from pathlib import Path

REQUIRED_DATA_REL = Path("neural-networks-mlp/dataset.csv")
CANDIDATE_REPO_URLS = [
    "https://github.com/truejulosdu13/ai4chemistry-bootcamp.git",
    "https://github.com/julschleinitz/ai4chemistry-bootcamp.git",
]

candidate_tutorial_dirs = [
    Path.cwd(),
    Path("/content/ai4chemistry-bootcamp/tutorials"),
    Path("/content/ai4chemistry-bootcamp/website/tutorials"),
]

def _find_tutorial_dir(candidates):
    for d in candidates:
        if (d / REQUIRED_DATA_REL).exists():
            return d
    return None

tutorial_dir = _find_tutorial_dir(candidate_tutorial_dirs)

# If not found and running in Colab, clone automatically.
if tutorial_dir is None and "google.colab" in sys.modules:
    clone_root = Path("/content/ai4chemistry-bootcamp")
    if not clone_root.exists():
        clone_ok = False
        for repo_url in CANDIDATE_REPO_URLS:
            exit_code = os.system(f"git clone {repo_url} {clone_root}")
            if exit_code == 0:
                clone_ok = True
                break
        if not clone_ok:
            raise RuntimeError("Failed to clone bootcamp repository in Colab.")

    candidate_tutorial_dirs = [
        Path("/content/ai4chemistry-bootcamp/tutorials"),
        Path("/content/ai4chemistry-bootcamp/website/tutorials"),
        Path.cwd(),
    ]
    tutorial_dir = _find_tutorial_dir(candidate_tutorial_dirs)

if tutorial_dir is None:
    raise FileNotFoundError(
        "Could not find neural-networks-mlp/dataset.csv. "
        "Set your working directory to the tutorials folder containing neural-networks-mlp/. "
        f"Current directory: {Path.cwd()}"
    )

os.chdir(tutorial_dir)
print(f"Working directory set to: {Path.cwd()}")
print(f"Found data root at: {Path.cwd() / 'neural-networks-mlp'}")

In [ ]:
@dataclass
class PathsConfig:
    dataset_csv: str = "./neural-networks-mlp/dataset.csv"
    train_split_csv: str = "./neural-networks-mlp/splits/train.csv"
    val_split_csv: str = "./neural-networks-mlp/splits/val.csv"
    test_split_csv: str = "./neural-networks-mlp/splits/test.csv"
    schema_csv: str = "./neural-networks-mlp/leaderboard/results_schema.csv"
    helper_py: str = "./neural-networks-mlp/leaderboard/submit_payload.py"

@dataclass
class TutorialConfig:
    target_col: str = "target"
    split_version: str = "v1"
    metric_name_classification: str = "roc_auc"
    metric_name_regression: str = "rmse"
    notebook_version: str = "nn_tutorial_v1"
    student_or_team: str = "Team XX"

PATHS = PathsConfig()
CFG = TutorialConfig()

def validate_required_files(paths: PathsConfig):
    required = [
        paths.dataset_csv,
        paths.train_split_csv,
        paths.val_split_csv,
        paths.test_split_csv,
        paths.schema_csv,
        paths.helper_py,
    ]
    missing = [p for p in required if not Path(p).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required tutorial files:\n"
            + "\n".join(f"- {p}" for p in missing)
            + f"\n\nCurrent working directory: {Path.cwd()}"
        )

validate_required_files(PATHS)
print(PATHS)
print(CFG)

In [ ]:
def _read_split_ids(path):
    split_df = pd.read_csv(path)
    if "sample_id" not in split_df.columns:
        raise ValueError(f"Split file {path} must contain a 'sample_id' column")
    return set(split_df["sample_id"].astype(str).tolist())

def load_fixed_splits(dataset_csv, train_csv, val_csv, test_csv, target_col):
    df = pd.read_csv(dataset_csv)

    required_cols = {"sample_id", target_col}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Dataset is missing required columns: {sorted(missing_cols)}")

    df["sample_id"] = df["sample_id"].astype(str)
    train_ids = _read_split_ids(train_csv)
    val_ids = _read_split_ids(val_csv)
    test_ids = _read_split_ids(test_csv)

    if train_ids & val_ids or train_ids & test_ids or val_ids & test_ids:
        raise ValueError("Split overlap detected. Splits must be disjoint.")

    all_split_ids = train_ids | val_ids | test_ids
    dataset_ids = set(df["sample_id"].tolist())
    missing_from_data = all_split_ids - dataset_ids
    if missing_from_data:
        raise ValueError(f"Split IDs not present in dataset: {list(missing_from_data)[:5]}")

    feature_cols = [c for c in df.columns if c.startswith("fp_")]
    if not feature_cols:
        feature_cols = [c for c in df.columns if c not in ["sample_id", target_col]]

    if not feature_cols:
        raise ValueError("No feature columns found. Add 'fp_' columns or non-id/non-target numeric columns.")

    train_df = df[df["sample_id"].isin(train_ids)].copy()
    val_df = df[df["sample_id"].isin(val_ids)].copy()
    test_df = df[df["sample_id"].isin(test_ids)].copy()

    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        if split_df.empty:
            raise ValueError(f"{split_name} split is empty. Check split files.")

    x_train = train_df[feature_cols].values.astype(np.float32)
    x_val = val_df[feature_cols].values.astype(np.float32)
    x_test = test_df[feature_cols].values.astype(np.float32)

    y_train = train_df[target_col].values
    y_val = val_df[target_col].values
    y_test = test_df[target_col].values

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_val = scaler.transform(x_val)
    x_test = scaler.transform(x_test)

    return {
        "feature_cols": feature_cols,
        "train": (x_train, y_train, train_df["sample_id"].tolist()),
        "val": (x_val, y_val, val_df["sample_id"].tolist()),
        "test": (x_test, y_test, test_df["sample_id"].tolist()),
    }

split_data = load_fixed_splits(
    PATHS.dataset_csv,
    PATHS.train_split_csv,
    PATHS.val_split_csv,
    PATHS.test_split_csv,
    CFG.target_col,
)

x_train, y_train, train_ids = split_data["train"]
x_val, y_val, val_ids = split_data["val"]
x_test, y_test, test_ids = split_data["test"]

print("Loaded fixed splits:")
print(f"  train: {x_train.shape}, val: {x_val.shape}, test: {x_test.shape}")
print(f"  number of features: {len(split_data['feature_cols'])}")

In [ ]:
def infer_task_type(y):
    unique = np.unique(y)
    if len(unique) <= 2 and set(unique).issubset({0, 1}):
        return "classification"
    return "regression"

TASK_TYPE = infer_task_type(y_train)
print(f"Inferred task type: {TASK_TYPE}")

if TASK_TYPE == "classification":
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
else:
    y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_val_t = torch.tensor(x_val, dtype=torch.float32)
x_test_t = torch.tensor(x_test, dtype=torch.float32)

In [ ]:
def make_loaders(batch_size):
    train_ds = TensorDataset(x_train_t, y_train_t)
    val_ds = TensorDataset(x_val_t, y_val_t)
    test_ds = TensorDataset(x_test_t, y_test_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [ ]:
class MolecularMLP(nn.Module):
    def __init__(self, in_dim, hidden_layers, hidden_width, dropout, activation):
        super().__init__()
        act_map = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "tanh": nn.Tanh,
        }
        if activation not in act_map:
            raise ValueError(f"Unsupported activation: {activation}")

        layers = []
        prev_dim = in_dim
        for _ in range(hidden_layers):
            layers.append(nn.Linear(prev_dim, hidden_width))
            layers.append(act_map[activation]())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_width

        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def metric_score(y_true, y_pred, task_type):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    if task_type == "classification":
        return roc_auc_score(y_true, y_pred)
    return np.sqrt(mean_squared_error(y_true, y_pred))

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, task_type):
    model.eval()
    total_loss = 0.0
    y_true = []
    y_pred = []

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)

        if task_type == "classification":
            probs = torch.sigmoid(logits)
            y_pred.extend(probs.cpu().numpy().reshape(-1).tolist())
        else:
            y_pred.extend(logits.cpu().numpy().reshape(-1).tolist())

        y_true.extend(yb.cpu().numpy().reshape(-1).tolist())

    avg_loss = total_loss / len(loader.dataset)
    score = metric_score(y_true, y_pred, task_type)
    return avg_loss, score, np.asarray(y_true), np.asarray(y_pred)

In [ ]:
def run_experiment(config):
    train_loader, val_loader, test_loader = make_loaders(config["batch_size"])

    model = MolecularMLP(
        in_dim=x_train.shape[1],
        hidden_layers=config["hidden_layers"],
        hidden_width=config["hidden_width"],
        dropout=config["dropout"],
        activation=config["activation"],
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss() if TASK_TYPE == "classification" else nn.MSELoss()
    if config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"],
        )
    elif config["optimizer"] == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=config["learning_rate"],
            momentum=0.9,
            weight_decay=config["weight_decay"],
        )
    else:
        raise ValueError(f"Unsupported optimizer: {config['optimizer']}")

    best_state = None
    best_val_score = -np.inf if TASK_TYPE == "classification" else np.inf
    no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_score": []}

    start_time = time.time()

    for epoch in range(config["epochs"]):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_score, _, _ = evaluate(model, val_loader, criterion, TASK_TYPE)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_score"].append(val_score)

        improved = (val_score > best_val_score) if TASK_TYPE == "classification" else (val_score < best_val_score)
        if improved:
            best_val_score = val_score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= config["early_stop_patience"]:
            break

    train_time_sec = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_score, y_true_test, y_pred_test = evaluate(model, test_loader, criterion, TASK_TYPE)

    result = {
        "model": model,
        "history": history,
        "train_time_sec": train_time_sec,
        "best_val_score": best_val_score,
        "test_score": test_score,
        "test_loss": test_loss,
        "y_true_test": y_true_test,
        "y_pred_test": y_pred_test,
        "epochs_trained": len(history["train_loss"]),
        "param_count": count_params(model),
    }
    return result

In [ ]:
def plot_learning_curves(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history["train_loss"], label="train")
    ax[0].plot(history["val_loss"], label="val")
    ax[0].set_title("Loss curves")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("Loss")
    ax[0].legend()

    ax[1].plot(history["val_score"], label="val score", color="tab:orange")
    ax[1].set_title("Validation metric")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Score")
    ax[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Starter hyperparameter sweep. Add or edit configurations for your group.
search_space = [
    {"hidden_layers": 2, "hidden_width": 128, "activation": "relu", "dropout": 0.1, "optimizer": "adam", "learning_rate": 1e-3, "batch_size": 128, "weight_decay": 1e-4, "epochs": 40, "early_stop_patience": 8},
    {"hidden_layers": 3, "hidden_width": 256, "activation": "relu", "dropout": 0.2, "optimizer": "adam", "learning_rate": 1e-3, "batch_size": 128, "weight_decay": 1e-4, "epochs": 40, "early_stop_patience": 8},
    {"hidden_layers": 4, "hidden_width": 256, "activation": "gelu", "dropout": 0.3, "optimizer": "adam", "learning_rate": 5e-4, "batch_size": 256, "weight_decay": 5e-4, "epochs": 50, "early_stop_patience": 10},
]

all_runs = []

for i, hp in enumerate(search_space, start=1):
    print(f"\nRun {i}/{len(search_space)}: {hp}")
    result = run_experiment(hp)
    metric_name = CFG.metric_name_classification if TASK_TYPE == "classification" else CFG.metric_name_regression
    run_row = {
        "run_idx": i,
        "hidden_layers": hp["hidden_layers"],
        "hidden_width": hp["hidden_width"],
        "activation": hp["activation"],
        "dropout": hp["dropout"],
        "optimizer": hp["optimizer"],
        "learning_rate": hp["learning_rate"],
        "batch_size": hp["batch_size"],
        "weight_decay": hp["weight_decay"],
        "epochs_trained": result["epochs_trained"],
        "param_count": result["param_count"],
        "train_time_sec": result["train_time_sec"],
        "best_val_score": result["best_val_score"],
        "test_score": result["test_score"],
        "metric_name": metric_name,
        "history": result["history"],
        "model": result["model"],
        "y_pred_test": result["y_pred_test"],
        "y_true_test": result["y_true_test"],
    }
    all_runs.append(run_row)

    print("  epochs:      ", run_row["epochs_trained"])
    print("  train_time_s:", round(run_row["train_time_sec"], 2))
    print("  val score:   ", round(run_row["best_val_score"], 4))
    print("  test score:  ", round(run_row["test_score"], 4))

runs_df = pd.DataFrame([{k: v for k, v in r.items() if k not in {"history", "model", "y_pred_test", "y_true_test"}} for r in all_runs])
runs_df

In [ ]:
# Select best model by validation score only.
if TASK_TYPE == "classification":
    best_idx = int(runs_df["best_val_score"].idxmax())
else:
    best_idx = int(runs_df["best_val_score"].idxmin())

best_run = all_runs[best_idx]
print("Selected run by validation metric:")
print(runs_df.loc[best_idx])

plot_learning_curves(best_run["history"], title="Best model learning curves")

In [ ]:
# Build a standardized submission payload for leaderboard upload.
def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

dataset_name = "tox21_fingerprints"
task_type_value = "multi_label_classification" if TASK_TYPE == "classification" else "regression"
metric_name = CFG.metric_name_classification if TASK_TYPE == "classification" else CFG.metric_name_regression

run_id = f"{CFG.student_or_team.lower().replace(' ', '_')}_{int(time.time())}"
payload = {
    "run_id": run_id,
    "timestamp_utc": utc_now_iso(),
    "student_or_team": CFG.student_or_team,
    "dataset_name": dataset_name,
    "task_type": task_type_value,
    "split_version": CFG.split_version,
    "model_family": "MLP",
    "hidden_layers": int(best_run["hidden_layers"]),
    "hidden_width": int(best_run["hidden_width"]),
    "activation": best_run["activation"],
    "dropout": float(best_run["dropout"]),
    "optimizer": best_run["optimizer"],
    "learning_rate": float(best_run["learning_rate"]),
    "batch_size": int(best_run["batch_size"]),
    "weight_decay": float(best_run["weight_decay"]),
    "epochs_trained": int(best_run["epochs_trained"]),
    "param_count": int(best_run["param_count"]),
    "train_time_sec": float(best_run["train_time_sec"]),
    "best_val_score": float(best_run["best_val_score"]),
    "test_score": float(best_run["test_score"]),
    "metric_name": metric_name,
    "device": str(DEVICE),
    "seed": int(SEED),
    "notebook_version": CFG.notebook_version,
    "notes": "fixed_split_final_submission",
}

print(json.dumps(payload, indent=2))

In [ ]:
# Write local submission artifacts.
# These files can be uploaded to Drive or sent to an Apps Script endpoint.

import importlib.util

helper_path = Path(PATHS.helper_py)
if not helper_path.exists():
    raise FileNotFoundError(f"Leaderboard helper not found: {helper_path}")

spec = importlib.util.spec_from_file_location("submit_payload", helper_path)
submit_payload = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submit_payload)

out_dir = Path("./neural-networks-mlp/leaderboard/submissions")
out_dir.mkdir(parents=True, exist_ok=True)
json_path = out_dir / f"{payload['run_id']}.json"
csv_log_path = out_dir / "local_leaderboard_log.csv"

submit_payload.write_payload_json(payload, json_path)
submit_payload.append_payload_csv(payload, csv_log_path)

print(f"Wrote JSON: {json_path}")
print(f"Appended CSV log: {csv_log_path}")

In [ ]:
# Optional: submit final payload to Apps Script endpoint.
# Set APPS_SCRIPT_URL to your deployed web app URL.

APPS_SCRIPT_URL = ""

if APPS_SCRIPT_URL:
    import requests

    resp = requests.post(APPS_SCRIPT_URL, json=payload, timeout=30)
    print("Status:", resp.status_code)
    print("Response:", resp.text[:500])
else:
    print("APPS_SCRIPT_URL is empty. Skipping remote submission.")

In [ ]:
# Visualize run efficiency vs performance for your local sweep (Pareto context).
plt.figure(figsize=(7, 5))
plt.scatter(runs_df["train_time_sec"], runs_df["test_score"], s=80)

for _, row in runs_df.iterrows():
    label = f"L{int(row['hidden_layers'])}-W{int(row['hidden_width'])}"
    plt.annotate(label, (row["train_time_sec"], row["test_score"]), fontsize=8)

plt.xlabel("Training time (seconds)")
plt.ylabel("Test score")
plt.title("Model efficiency vs held-out test performance")
plt.grid(alpha=0.25)
plt.show()